# SupportFlow: a Multi-Agent Support Team (AutoGen + local Ollama)

3 AI agents work as a team on support tickets, running **100% locally** (no API cost, private data):

| Agent | Job |
|---|---|
| `TriageSpecialist` | category + priority |
| `ResearchSpecialist` | cause + solution steps |
| `ResponseSpecialist` | the reply to the customer |

**Setup:** Ollama + `llama3.2` (3B) on an RTX 4090 · AG2 0.9.10 (classic `import autogen` API).

The lab's original code is kept as is. My changes are marked `[added]` or `[fix]`.

## Cell 1: Local AI environment setup
Pre-flight check: is Ollama installed and is `llama3.2` available?

In [1]:
import autogen
import json
from datetime import datetime
import subprocess
import time  # [added] measure real processing time

print("🎯 SupportFlow - AutoGen Customer Support Hub")
print("=" * 50)
print("Automating customer support with multi-agent collaboration")
print("✅ Using local Ollama with llama3.2 - No API keys required!")

# Check if Ollama is available and llama3.2 is installed
def check_ollama_setup():
    try:
        result = subprocess.run(['ollama', 'list'], capture_output=True, text=True)
        if result.returncode == 0:
            print("\n📋 Available Ollama models:")
            print(result.stdout)
            if 'llama3.2' in result.stdout:
                print("✅ llama3.2 is available and ready!")
                return True
            else:
                print("❌ llama3.2 not found. Install it with: ollama pull llama3.2")
                return False
        else:
            print(f"❌ Error checking Ollama: {result.stderr}")
            return False
    except FileNotFoundError:
        print("❌ Ollama not found. Please install: https://ollama.ai/download")
        return False

ollama_ready = check_ollama_setup()
if not ollama_ready:
    print("\n🚨 SETUP REQUIRED:")
    print("1. Install Ollama: https://ollama.ai/download")
    print("2. Pull llama3.2: ollama pull llama3.2")
    print("3. Start Ollama: ollama serve")
    print("4. Re-run this cell to verify")
else:
    print("\n🚀 Ready to build with local AI!")

🎯 SupportFlow - AutoGen Customer Support Hub
Automating customer support with multi-agent collaboration
✅ Using local Ollama with llama3.2 - No API keys required!

📋 Available Ollama models:
NAME               ID              SIZE      MODIFIED       
llama3.2:latest    a80c4f17acd5    2.0 GB    13 minutes ago    

✅ llama3.2 is available and ready!

🚀 Ready to build with local AI!


## Cell 2: AutoGen configuration
AutoGen talks to Ollama through its OpenAI-compatible endpoint, so no real API key is needed and the cost is zero.

In [2]:
# Configure AutoGen with local Ollama
config_list = [
    {
        "model": "llama3.2",
        "base_url": "http://localhost:11434/v1",
        "api_key": "ollama",  # Ollama doesn't need a real API key
        "temperature": 0.3,  # Lower temperature for consistent support responses
        "price": [0, 0]
    }
]

llm_config = {
    "config_list": config_list,
    "timeout": 120,
    "cache_seed": None,  # [added] disable AutoGen's response cache so every run (and timing) is real
}

print("🔧 AutoGen configuration ready")
print(f"Model: Ollama llama3.2 (local)")
print(f"Base URL: http://localhost:11434/v1")
print(f"Temperature: 0.3 (for consistent support responses)")
print(f"✅ No API costs - runs completely local!")
print(f"🔒 Privacy-first - all data stays on your machine")

🔧 AutoGen configuration ready
Model: Ollama llama3.2 (local)
Base URL: http://localhost:11434/v1
Temperature: 0.3 (for consistent support responses)
✅ No API costs - runs completely local!
🔒 Privacy-first - all data stays on your machine


## Cell 3: Triage Agent, the sorter
A strict system message defines the categories, the priority rules and a fixed output format that the next agent can rely on.

In [3]:
# 1. Triage Agent - Categorizes and prioritizes tickets
triage_agent = autogen.AssistantAgent(
    name="TriageSpecialist",
    system_message="""You are a Customer Support Triage Specialist for CloudTech Solutions.

    RESPONSIBILITIES:
    - Analyze customer tickets and categorize them
    - Assign priority levels (High/Medium/Low)
    - Identify the issue type (Technical, Billing, Account, Feature Request)

    CATEGORIES:
    - Technical: Login issues, API errors, performance problems
    - Billing: Payment questions, invoice issues, subscription changes
    - Account: Access problems, user management, permissions
    - Feature Request: New functionality requests, improvements

    PRIORITY LEVELS:
    - High: Service outages, payment failures, security issues
    - Medium: Feature questions, minor bugs, account updates
    - Low: General questions, documentation requests, suggestions

    Always respond in this format:
    Category: [Technical/Billing/Account/Feature Request]
    Priority: [High/Medium/Low]
    Reasoning: [Brief explanation]
    Next Action: [What type of research/response is needed]
    """,
    llm_config=llm_config
)

print("✅ Triage Agent created")
print("   🎯 Specializes in ticket categorization and prioritization")

✅ Triage Agent created
   🎯 Specializes in ticket categorization and prioritization


## Cell 4: Research Agent, the knowledge expert
The company knowledge base is embedded in the system message. In production this would be a retrieval tool connected to real documentation.

In [4]:
# 2. Research Agent - Finds solutions and relevant information
research_agent = autogen.AssistantAgent(
    name="ResearchSpecialist",
    system_message="""You are a Customer Support Research Specialist for CloudTech Solutions.

    RESPONSIBILITIES:
    - Research solutions for categorized customer issues
    - Reference CloudTech's knowledge base and documentation
    - Provide technical solutions and troubleshooting steps
    - Gather relevant information for response crafting

    KNOWLEDGE BASE (CloudTech Solutions):
    - API Documentation: https://docs.cloudtech.com/api
    - Billing Portal: https://billing.cloudtech.com
    - Status Page: https://status.cloudtech.com
    - User Guide: https://help.cloudtech.com
    - Contact: support@cloudtech.com, Phone: 1-800-CLOUD-99

    COMMON SOLUTIONS:
    - Login Issues: Check 2FA, password reset, browser cache
    - API Errors: Verify API key, check rate limits, review documentation
    - Billing: Direct to billing portal, provide contact for complex issues
    - Account Access: Verify permissions, check user roles

    Always provide:
    1. Root cause analysis
    2. Step-by-step solution
    3. Relevant documentation links
    4. Escalation path if needed
    """,
    llm_config=llm_config
)

print("✅ Research Agent created")
print("   🔍 Specializes in solution research and troubleshooting")

✅ Research Agent created
   🔍 Specializes in solution research and troubleshooting


## Cell 5: Response Agent, the communicator
Turns the research into a customer-ready reply with a fixed structure and brand tone.

In [5]:
# 3. Response Agent - Crafts professional customer responses
response_agent = autogen.AssistantAgent(
    name="ResponseSpecialist",
    system_message="""You are a Customer Support Response Specialist for CloudTech Solutions.

    RESPONSIBILITIES:
    - Craft professional, empathetic customer responses
    - Include all solution steps and relevant links
    - Maintain CloudTech's friendly, helpful tone
    - Ensure responses are complete and actionable

    RESPONSE GUIDELINES:
    - Start with empathy and acknowledgment
    - Provide clear, step-by-step solutions
    - Include relevant documentation links
    - Offer additional assistance
    - Professional but warm tone
    - Use CloudTech branding elements

    TEMPLATE STRUCTURE:
    1. Greeting and acknowledgment
    2. Solution steps (numbered if multiple)
    3. Additional resources
    4. Follow-up offer
    5. Professional closing

    Always ensure responses are:
    - Complete and actionable
    - Easy to understand
    - Include next steps
    - Maintain helpful tone
    """,
    llm_config=llm_config
)

print("✅ Response Agent created")
print("   ✍️ Specializes in professional customer communication")
print("\n🎉 Support agent team ready for action!")
print("   Team: Triage → Research → Response")
print("   Powered by: Local Ollama llama3.2")

✅ Response Agent created
   ✍️ Specializes in professional customer communication

🎉 Support agent team ready for action!
   Team: Triage → Research → Response
   Powered by: Local Ollama llama3.2


## Cell 6: Sequential workflow
Each agent receives the ticket plus the previous agent's output. The function returns the full audit trail of the ticket.

In [6]:
def process_support_ticket(customer_ticket):
    """Process a customer support ticket through our agent workflow"""

    start = time.perf_counter()  # [added]
    print(f"\n🎫 NEW SUPPORT TICKET")
    print("=" * 40)
    print(f"Customer Message: {customer_ticket}")
    print("-" * 40)

    # Step 1: Triage the ticket
    print("\n🎯 STEP 1: Ticket Triage (Local AI Processing...)")
    triage_response = triage_agent.generate_reply(
        messages=[{"role": "user", "content": f"Please triage this customer ticket: {customer_ticket}"}]
    )
    print(f"Triage Analysis: {triage_response}")

    # Step 2: Research solutions
    print("\n🔍 STEP 2: Solution Research (Local AI Processing...)")
    research_response = research_agent.generate_reply(
        messages=[
            {"role": "user", "content": f"Research solutions for this ticket: {customer_ticket}"},
            {"role": "assistant", "content": f"Triage analysis: {triage_response}"}
        ]
    )
    print(f"Research Results: {research_response}")

    # Step 3: Craft customer response
    print("\n✍️ STEP 3: Customer Response (Local AI Processing...)")
    customer_response = response_agent.generate_reply(
        messages=[
            {"role": "user", "content": f"Draft a professional response for: {customer_ticket}"},
            {"role": "assistant", "content": f"Research: {research_response}"}
        ]
    )
    print(f"Final Response: {customer_response}")

    elapsed = time.perf_counter() - start  # [added]
    print(f"\n⏱️ Processed in {elapsed:.1f} seconds")  # [added]
    return {
        "ticket": customer_ticket,
        "triage": triage_response,
        "research": research_response,
        "response": customer_response,
        "elapsed_seconds": round(elapsed, 1),  # [added]
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

print("🔧 Support ticket processing function ready")
print("   Workflow: Customer Ticket → Triage → Research → Response")
print("   AI Engine: Local Ollama llama3.2")

🔧 Support ticket processing function ready
   Workflow: Customer Ticket → Triage → Research → Response
   AI Engine: Local Ollama llama3.2


## Cell 7: Real customer scenario testing
Demo 1 is the original technical ticket. Demos 2 and 3 are added so that the business analysis is based on more than one ticket.

In [7]:
# Test Case 1: Technical Issue
print("🧪 DEMO 1: Technical Support Issue")
ticket1 = "I'm getting API error 401 when trying to authenticate. My API key worked yesterday but now it's failing. Can you help?"

result1 = process_support_ticket(ticket1)

🧪 DEMO 1: Technical Support Issue

🎫 NEW SUPPORT TICKET
Customer Message: I'm getting API error 401 when trying to authenticate. My API key worked yesterday but now it's failing. Can you help?
----------------------------------------

🎯 STEP 1: Ticket Triage (Local AI Processing...)


Triage Analysis: Category: Technical
Priority: Medium
Reasoning: The customer is experiencing an API error (401 Unauthorized) when trying to authenticate, but their API key previously worked. This suggests a potential issue with the API credentials or the authentication process.
Next Action: Research: Review CloudTech Solutions' API documentation and troubleshooting guides to identify possible causes of the 401 error, such as expired or invalid API keys, rate limiting, or changes to the authentication endpoint.

🔍 STEP 2: Solution Research (Local AI Processing...)


Research Results: 

✍️ STEP 3: Customer Response (Local AI Processing...)


Final Response: 401 Unauthorized API error

Dear [Customer Name],

Thank you for reaching out to CloudTech Solutions. Sorry to hear that you're experiencing issues with API authentication. I'm here to help you troubleshoot the problem.

First, let's try to resolve the issue. API errors can be caused by various factors, such as changes in your API key or the API endpoint itself. Since your API key worked previously, it's likely that there's a temporary issue with the API.

Here are some steps to help you resolve the 401 Unauthorized API error:

1. **Check your API key**: Ensure that your API key is still active and hasn't expired. You can check the expiration date on the CloudTech dashboard.
2. **Verify API endpoint**: Double-check that you're using the correct API endpoint and that it's not being blocked by any intermediate proxies or firewalls.
3. **Clear cache and cookies**: Clear your browser's cache and cookies, and try authenticating again. This can help resolve issues related to 

In [8]:
# [added] Test Case 2: Billing Issue
print("🧪 DEMO 2: Billing Issue")
ticket2 = "My credit card was declined when renewing our annual subscription and now our account shows a payment failure warning. We can't afford any downtime. What should we do?"

result2 = process_support_ticket(ticket2)

🧪 DEMO 2: Billing Issue

🎫 NEW SUPPORT TICKET
Customer Message: My credit card was declined when renewing our annual subscription and now our account shows a payment failure warning. We can't afford any downtime. What should we do?
----------------------------------------

🎯 STEP 1: Ticket Triage (Local AI Processing...)


Triage Analysis: Category: Billing
Priority: High
Reasoning: The customer is experiencing a payment failure warning, which indicates a critical issue that could cause downtime. The payment failure warning is a high-priority issue that requires immediate attention to resolve the customer's concern.
Next Action: Research: Check if the payment failure is due to a declined card, expired card, or other billing-related issues. Also, verify if the subscription renewal was successful or if there are any other errors that may have caused the payment failure warning.

🔍 STEP 2: Solution Research (Local AI Processing...)


Research Results: 

✍️ STEP 3: Customer Response (Local AI Processing...)


Final Response:  I've checked on the status of your renewal and found that the issue is indeed related to the credit card decline. I'm here to help you resolve this as quickly as possible.

Solution Steps:

1.  **Contact your bank**: Reach out to your credit card issuer to confirm that the card is declined due to insufficient funds or any other reason. They may be able to provide additional information or assistance.
2.  **Update your payment method**: Once you've resolved the issue with your bank, update your payment method in your CloudTech account to a new credit card or another accepted payment method.
3.  **Renew your subscription**: After updating your payment method, you can try renewing your subscription again. If you're still experiencing issues, please let me know, and I'll be happy to help you troubleshoot further.

Additional Resources:

*   [Payment Method Update Guide](link to guide): This guide provides step-by-step instructions on updating your payment method in your Cl

In [9]:
# [added] Test Case 3: Account / Feature Request
print("🧪 DEMO 3: Account & Feature Question")
ticket3 = "How do I give a new teammate read-only access to our dashboards? Also, is there any plan to add dark mode?"

result3 = process_support_ticket(ticket3)

🧪 DEMO 3: Account & Feature Question

🎫 NEW SUPPORT TICKET
Customer Message: How do I give a new teammate read-only access to our dashboards? Also, is there any plan to add dark mode?
----------------------------------------

🎯 STEP 1: Ticket Triage (Local AI Processing...)


Triage Analysis: Category: Account
Priority: Medium
Reasoning: The customer is requesting access control for a new teammate, which is an account-related issue, and also inquiring about a feature request (dark mode) that is not a critical issue.
Next Action: Research the process for granting read-only access to dashboards and gather information on the current status of the dark mode feature request.

🔍 STEP 2: Solution Research (Local AI Processing...)
Research Results: 

✍️ STEP 3: Customer Response (Local AI Processing...)


Final Response:  I've checked our documentation and couldn't find any information on adding dark mode. I'd be happy to provide you with an update on this feature, but I couldn't find any official announcement.

Response:

Dear [Customer Name],

Thank you for reaching out to us about giving your new teammate read-only access to our dashboards. I'm happy to help you with this request.

To grant your new teammate read-only access to our dashboards, please follow these steps:

1. Log in to your CloudTech account and navigate to the "Users" section.
2. Click on the "Add User" button and enter your new teammate's email address.
3. Select the "Read-only" permission level for the user.
4. Assign the user to the relevant dashboards by clicking on the "Assign" button.
5. Make sure to update the user's permissions to ensure they can only view the data, not edit it.

For more detailed instructions, you can refer to our documentation on user management: [https://cloudtech.solutions/user-management/

## Cell 8: Multi-agent collaboration (group chat)
A `GroupChatManager` coordinates the three agents on a ticket that has four issues at once. The manager uses the LLM to decide who speaks next.

In [10]:
def collaborative_support_demo():
    """Demonstrate agents collaborating on a complex multi-faceted issue"""

    complex_ticket = """
    Hi CloudTech Support,

    We're having multiple issues:
    1. Our API calls are returning 500 errors intermittently
    2. We were double-charged on our invoice this month
    3. We need to add 5 new team members but can't access user management
    4. Is there a way to get real-time status updates?

    This is urgent as it's affecting our production system.

    Thanks,
    Sarah Chen, CTO at TechStartup Inc
    """

    print("🚨 COMPLEX MULTI-ISSUE TICKET")
    print("=" * 40)
    print(complex_ticket)
    print("=" * 40)

    # Create a group chat for agent collaboration
    groupchat = autogen.GroupChat(
        agents=[triage_agent, research_agent, response_agent],
        messages=[],
        max_round=8
    )

    manager = autogen.GroupChatManager(groupchat=groupchat, llm_config=llm_config)

    # Start collaborative resolution
    print("\n👥 AGENTS COLLABORATING WITH LOCAL AI...")
    print("Watch how the agents work together using Ollama llama3.2:")

    triage_agent.initiate_chat(
        manager,
        message=f"""We have a complex customer ticket with multiple issues.
        Let's work together to provide a comprehensive solution.

        Customer Ticket: {complex_ticket}

        Triage Agent: Please categorize and prioritize each issue.
        Research Agent: Find solutions for each identified problem.
        Response Agent: Craft a comprehensive response addressing all issues.
        """,
        max_turns=6
    )
    return groupchat  # [added] keep the transcript

# Run the collaborative demo
print("🔄 Starting multi-agent collaboration demo...")
collab_start = time.perf_counter()  # [added]
try:  # [added] report the error instead of stopping
    groupchat = collaborative_support_demo()
    collab_error = None
except Exception as exc:
    groupchat, collab_error = None, f"{type(exc).__name__}: {str(exc)[:120]}"
collab_seconds = time.perf_counter() - collab_start  # [added]
if collab_error:
    print(f"\n⚠️ Original group chat FAILED after {collab_seconds:.1f} s -> {collab_error}")
else:
    print(f"\n⏱️ Group chat finished in {collab_seconds:.1f} seconds, {len(groupchat.messages)} messages")

🔄 Starting multi-agent collaboration demo...
🚨 COMPLEX MULTI-ISSUE TICKET

    Hi CloudTech Support,

    We're having multiple issues:
    1. Our API calls are returning 500 errors intermittently
    2. We were double-charged on our invoice this month
    3. We need to add 5 new team members but can't access user management
    4. Is there a way to get real-time status updates?

    This is urgent as it's affecting our production system.

    Thanks,
    Sarah Chen, CTO at TechStartup Inc
    

👥 AGENTS COLLABORATING WITH LOCAL AI...
Watch how the agents work together using Ollama llama3.2:
TriageSpecialist (to chat_manager):



We have a complex customer ticket with multiple issues.
        Let's work together to provide a comprehensive solution.

        Customer Ticket: 
    Hi CloudTech Support,

    We're having multiple issues:
    1. Our API calls are returning 500 errors intermittently
    2. We were double-charged on our invoice this month
    3. We need to add 5 new team members but can't access user management
    4. Is there a way to get real-time status updates?

    This is urgent as it's affecting our production system.

    Thanks,
    Sarah Chen, CTO at TechStartup Inc
    

        Triage Agent: Please categorize and prioritize each issue.
        Research Agent: Find solutions for each identified problem.
        Response Agent: Craft a comprehensive response addressing all issues.
        



--------------------------------------------------------------------------------



Next speaker: TriageSpecialist



TriageSpecialist (to chat_manager):




--------------------------------------------------------------------------------


... (167 outputs left out)


--------------------------------------------------------------------------------



>>>>>>>> TERMINATING RUN (50045ca4-ecee-42bf-aabe-1e1b5a7bd649): Maximum rounds (8) reached



>>>>>>>> TERMINATING RUN (1b8c9cc9-31ba-4f6e-9340-f709edf3e1dd): Maximum turns (6) reached



⏱️ Group chat finished in 426.1 seconds, 48 messages


## Cell 8b [fix]: Fixing the two bugs
1. **Empty research answers.** The previous agent's output was sent with role `assistant`, so the model thought it had already answered. **Fix:** send it inside a `user` message.
2. **The group chat looped.** A 3B model chose the next speaker badly and the history outgrew the 4,096-token context. **Fix:** a fixed turn order (`round_robin`) + a `max_tokens` limit.

In [11]:
def process_support_ticket_v2(customer_ticket):
    """Same 3-agent pipeline, but each agent gets its context in a single user message."""
    start = time.perf_counter()
    print(f"\n🎫 TICKET: {customer_ticket}")

    triage = triage_agent.generate_reply(messages=[
        {"role": "user", "content": f"Please triage this customer ticket: {customer_ticket}"}])
    print(f"\n🎯 TRIAGE:\n{triage}")

    research = research_agent.generate_reply(messages=[
        {"role": "user", "content": f"Customer ticket: {customer_ticket}\n\nTriage analysis:\n{triage}\n\n"
                                    "Research solutions for this ticket."}])
    print(f"\n🔍 RESEARCH:\n{research}")

    response = response_agent.generate_reply(messages=[
        {"role": "user", "content": f"Customer ticket: {customer_ticket}\n\nResearch notes:\n{research}\n\n"
                                    "Draft the reply to the customer."}])
    print(f"\n✍️ FINAL RESPONSE:\n{response}")

    elapsed = time.perf_counter() - start
    print(f"\n⏱️ Processed in {elapsed:.1f} seconds")
    return {"ticket": customer_ticket, "triage": triage, "research": research, "response": response,
            "elapsed_seconds": round(elapsed, 1), "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

fixed_results = [process_support_ticket_v2(t) for t in (ticket1, ticket2, ticket3)]


🎫 TICKET: I'm getting API error 401 when trying to authenticate. My API key worked yesterday but now it's failing. Can you help?



🎯 TRIAGE:
Category: Technical
Priority: Medium
Reasoning: The customer is experiencing an API error 401, which typically indicates an authentication issue. Although their API key worked previously, the current failure suggests a potential change in the API's behavior or an issue with the key itself. The customer is seeking assistance with a technical issue.

Next Action: Research the API error 401 and its possible causes, such as changes to the API's authentication mechanism or key requirements.



🔍 RESEARCH:
**Research and Analysis**

After researching the API error 401, I found that this error typically indicates an authentication issue, where the API is unable to verify the user's credentials. Given that the customer's API key worked previously, it's possible that there's a change in the API's behavior or an issue with the key itself.

Possible causes for this error include:

1. **API Key Rotation**: CloudTech Solutions may have rotated the API key, which would require the customer to obtain a new key.
2. **Changes to Authentication Mechanism**: There might be a change in the authentication mechanism, such as a new security protocol or updated rate limiting rules.
3. **Key Expiration**: The API key may have expired, which would require the customer to obtain a new key.
4. **Rate Limiting**: The API may have been rate limited, which would prevent the customer from making too many requests within a certain timeframe.

**Step-by-Step Solution**

To resolve this issue, I recomme


✍️ FINAL RESPONSE:
Here's a draft reply to the customer:

"Dear [Customer Name],

Thank you for reaching out to CloudTech Solutions Support. We're here to help you resolve the API error 401 issue.

We've checked the API key status, and it appears that the key has not been rotated recently. However, we recommend verifying the API key format and ensuring that it meets the requirements specified in our API documentation: https://docs.cloudtech.com/api.

To confirm, please check the following:

* Is the API key in the correct format?
* Is the API key up-to-date and not expired?

If you're still experiencing issues, please provide us with the following information:
	* API key (if applicable)
	* Request details (e.g., API endpoint, method, headers)
	* Any relevant logs or error messages

We'll do our best to assist you in resolving this issue. If you need further assistance, please don't hesitate to contact us.

Best regards,
[Your Name]
CloudTech Solutions Support"

I made a few changes to


🎯 TRIAGE:
Category: Billing
Priority: High
Reasoning: The customer is experiencing a payment failure warning, which is causing a potential downtime issue. The customer is also expressing concern about the impact on their business, indicating a high level of urgency.
Next Action: Research the customer's payment method and subscription details to determine the cause of the payment failure and assess the impact on their account.



🔍 RESEARCH:
**Root Cause Analysis:**

Based on the customer's issue, I've identified the following potential root causes:

1. **Invalid or expired credit card**: The customer's credit card may have been declined due to an invalid or expired card number.
2. **Insufficient funds**: The customer's account balance may be insufficient to cover the subscription renewal fee.
3. **Payment method issues**: The customer's payment method may be incorrect or not recognized by our system.

**Step-by-Step Solution:**

To resolve this issue, I recommend the following steps:

1. **Verify payment method**: Check the customer's payment method to ensure it's correct and up-to-date. If the payment method is incorrect, ask the customer to update their payment information.
2. **Check account balance**: Verify the customer's account balance to ensure it's sufficient to cover the subscription renewal fee. If the balance is insufficient, discuss possible payment options with the customer.
3. **Review subscrip


✍️ FINAL RESPONSE:
Here's a draft reply to the customer:

"Dear [Customer Name],

Thank you for your prompt response and for providing the necessary information to help resolve the payment failure warning on your account. I've verified your payment method and account balance, and everything appears to be in order.

To confirm, I've checked your subscription details, and there are no issues with the subscription itself. However, I did notice that your account balance is slightly below the required amount to cover the subscription renewal fee. I'd like to discuss possible payment options with you to ensure you can continue to use our services without any downtime.

Please let me know if you're able to make the payment today, or if you'd like to explore alternative payment arrangements. I'm here to help and want to ensure you can continue to rely on our services.

Additionally, I've updated your payment information in our system, and you should no longer receive the payment failure warni


🎯 TRIAGE:
Category: Account
Priority: Medium
Reasoning: The customer is requesting access for a new teammate, which is an account-related issue, and also inquiring about a feature request (dark mode).
Next Action: Research the process for granting read-only access to dashboards and gather information on the current status of the dark mode feature request to provide a more informed response.



🔍 RESEARCH:
Based on the triage analysis, I've researched the solutions for granting read-only access to dashboards and gathered information on the current status of the dark mode feature request.

**Granting Read-Only Access to Dashboards:**

To grant a new teammate read-only access to our dashboards, I recommend the following steps:

1. **Log in to the Admin Dashboard**: Go to the Admin Dashboard (https://admin.cloudtech.com) and log in with the admin credentials.
2. **Navigate to Users**: Click on the "Users" tab and select the user account for the new teammate.
3. **Edit User Settings**: Click on the "Edit" button next to the user account and navigate to the "User Settings" section.
4. **Select Access Level**: Under "Access Level," select "Read-Only" from the dropdown menu.
5. **Save Changes**: Click "Save" to apply the changes.

Additionally, I recommend that the admin also:

* **Verify 2FA**: Ensure that the new teammate has 2-factor authentication (2FA) enabled to maintain secu


✍️ FINAL RESPONSE:
Here's a draft reply to the customer:

Subject: Re: Granting Read-Only Access to Dashboards and Dark Mode Feature Request

Dear [Customer Name],

Thank you for reaching out to us about granting your new teammate read-only access to our dashboards and inquiring about the status of our dark mode feature request. We apologize for any inconvenience this may have caused and are happy to assist you with both issues.

To grant your new teammate read-only access to our dashboards, please follow these steps:

1. Log in to the Admin Dashboard (https://admin.cloudtech.com) and log in with your admin credentials.
2. Navigate to Users and select the user account for your new teammate.
3. Edit User Settings and select "Read-Only" from the dropdown menu under Access Level.
4. Save Changes and verify that the new teammate has 2-factor authentication (2FA) enabled to maintain security.
5. Test the new teammate's access to ensure they can view the dashboards correctly.

Regarding the

In [12]:
complex_ticket = """Hi CloudTech Support,
We're having multiple issues:
1. Our API calls are returning 500 errors intermittently
2. We were double-charged on our invoice this month
3. We need to add 5 new team members but can't access user management
4. Is there a way to get real-time status updates?
This is urgent as it's affecting our production system.
Thanks, Sarah Chen, CTO at TechStartup Inc"""

# Limit reply length (max_tokens) and allow more time per reply
bounded_llm_config = {"config_list": [{**config_list[0], "max_tokens": 700}], "timeout": 300}
for agent in (triage_agent, research_agent, response_agent):
    agent.llm_config = bounded_llm_config  # applies to these agents from now on

customer = autogen.UserProxyAgent(name="Customer", human_input_mode="NEVER",
                                  code_execution_config=False, default_auto_reply="")

groupchat_v2 = autogen.GroupChat(
    agents=[customer, triage_agent, research_agent, response_agent],
    messages=[],
    max_round=4,                            # customer message + one turn per specialist
    speaker_selection_method="round_robin", # fixed order instead of LLM-chosen speakers
)
manager_v2 = autogen.GroupChatManager(groupchat=groupchat_v2, llm_config=bounded_llm_config)

collab_v2_start = time.perf_counter()
customer.initiate_chat(manager_v2, message=complex_ticket)
collab_v2_seconds = time.perf_counter() - collab_v2_start
print(f"\n⏱️ Fixed group chat finished in {collab_v2_seconds:.1f} s")
print("Speaker order:", " → ".join(m.get("name", "?") for m in groupchat_v2.messages))

Customer (to chat_manager):



Hi CloudTech Support,
We're having multiple issues:
1. Our API calls are returning 500 errors intermittently
2. We were double-charged on our invoice this month
3. We need to add 5 new team members but can't access user management
4. Is there a way to get real-time status updates?
This is urgent as it's affecting our production system.
Thanks, Sarah Chen, CTO at TechStartup Inc



--------------------------------------------------------------------------------



Next speaker: TriageSpecialist



TriageSpecialist (to chat_manager):



Category: [Technical/Billing/Account/Feature Request]
Priority: [High/Medium/Low]
Reasoning: 
- API errors (500) are a critical issue, as they can impact production systems. High priority.
- Double-charge on invoice is a billing issue, but it's not as critical as the API errors. Medium priority.
- Access issues with user management is an account issue, but it's not as urgent as the API errors. Medium priority.
- Real-time status updates is a feature request, but it's not a critical issue. Low priority.

Next Action: 
- For the API errors (500), I need to research the possible causes and potential solutions. Can you please provide more information about the API calls that are failing, such as the specific endpoints and any error messages?
- For the double-charge on invoice, I need to investigate the billing process and see if there's an error that caused the double charge. Can you please provide your invoice number and a copy of the invoice?
- For the access issues with user management,

... (7 outputs left out)

Dear Sarah Chen, CTO at TechStartup Inc,

Thank you for reaching out to CloudTech Support regarding the issues you're experiencing with our services. We apologize for the inconvenience and are committed to resolving these issues as soon as possible.

Below are the next steps we'll take to address each of your concerns:

**API Errors (500)**

To help us better understand the issue, we need more information about the specific API calls that are failing. Could you please provide the following:

* The exact API endpoints that are returning 500 errors
* Any error messages or codes associated with the 500 errors
* The frequency and duration of the errors (e.g., are they intermittent, or do they occur consistently?)

We've reviewed our API documentation (https://docs.cloudtech.com/api) and checked the status page (https://status.cloudtech.com) to ensure there are no known issues. However, we'd like to investigate further to identify the root cause of the problem.

**Double-Charge on Invoice**


--------------------------------------------------------------------------------



>>>>>>>> TERMINATING RUN (77085003-634c-41f4-8a38-3eaa938e6c20): Maximum rounds (4) reached



⏱️ Fixed group chat finished in 31.5 s
Speaker order: Customer → TriageSpecialist → ResearchSpecialist → ResponseSpecialist


## Cell 9: Business impact analysis (original code)
This is the lab's original calculation, using its assumed values (45 seconds per ticket, 30 minutes manual, $35/hour).

In [13]:
def analyze_support_performance():
    """Calculate the business impact of our automated support system"""

    # Simulate processing metrics
    tickets_processed = 4  # Including the complex multi-issue ticket
    avg_processing_time = 45  # seconds per ticket (vs 30 minutes manual)

    print("📈 SUPPORTFLOW PERFORMANCE ANALYSIS")
    print("=" * 45)

    # Current automation results
    print(f"✅ Tickets Processed: {tickets_processed}")
    print(f"⚡ Avg Processing Time: {avg_processing_time} seconds")
    print(f"🤖 AI Engine: Local Ollama llama3.2")
    print(f"🎯 Success Rate: 100% (all tickets properly triaged and responded)")

    # Business impact calculation
    manual_time_per_ticket = 30  # minutes
    automated_time_per_ticket = 0.75  # 45 seconds in minutes
    time_saved_per_ticket = manual_time_per_ticket - automated_time_per_ticket

    daily_tickets = 150
    daily_time_saved = (time_saved_per_ticket * daily_tickets) / 60  # hours
    annual_time_saved = daily_time_saved * 250  # work days

    support_hourly_rate = 35  # Average support specialist rate
    annual_cost_savings = annual_time_saved * support_hourly_rate

    # Response time improvement
    manual_response_time = 300  # 5 hours average
    automated_response_time = 5   # 5 minutes
    response_improvement = ((manual_response_time - automated_response_time) / manual_response_time) * 100

    print(f"\n💰 BUSINESS IMPACT CALCULATION")
    print("=" * 35)
    print(f"Time Saved Per Ticket: {time_saved_per_ticket:.1f} minutes")
    print(f"Daily Time Saved: {daily_time_saved:.1f} hours")
    print(f"Annual Hours Saved: {annual_time_saved:,.0f} hours")
    print(f"Annual Cost Savings: ${annual_cost_savings:,.2f}")
    print(f"Response Time Improvement: {response_improvement:.0f}%")
    print(f"ROI: {annual_cost_savings / 25000:.1f}x (assuming $25K implementation)")
    print(f"💸 Additional Benefit: Zero ongoing API costs with local Ollama!")

    # Customer satisfaction impact
    print(f"\n😊 CUSTOMER EXPERIENCE IMPROVEMENTS")
    print("=" * 40)
    print(f"✅ 24/7 Support Availability (vs business hours only)")
    print(f"✅ Consistent Response Quality (no agent variation)")
    print(f"✅ Immediate Acknowledgment (vs hours of waiting)")
    print(f"✅ Comprehensive Multi-Issue Handling")
    print(f"🔒 Privacy-First: All data stays local with Ollama")
    print(f"💰 Cost-Effective: No per-token charges or usage limits")

    return {
        "daily_time_saved_hours": daily_time_saved,
        "annual_cost_savings": annual_cost_savings,
        "response_time_improvement": response_improvement,
        "roi_multiple": annual_cost_savings / 25000
    }

# Calculate and display business impact
impact_metrics = analyze_support_performance()

📈 SUPPORTFLOW PERFORMANCE ANALYSIS
✅ Tickets Processed: 4
⚡ Avg Processing Time: 45 seconds
🤖 AI Engine: Local Ollama llama3.2
🎯 Success Rate: 100% (all tickets properly triaged and responded)

💰 BUSINESS IMPACT CALCULATION
Time Saved Per Ticket: 29.2 minutes
Daily Time Saved: 73.1 hours
Annual Hours Saved: 18,281 hours
Annual Cost Savings: $639,843.75
Response Time Improvement: 98%
ROI: 25.6x (assuming $25K implementation)
💸 Additional Benefit: Zero ongoing API costs with local Ollama!

😊 CUSTOMER EXPERIENCE IMPROVEMENTS
✅ 24/7 Support Availability (vs business hours only)
✅ Consistent Response Quality (no agent variation)
✅ Immediate Acknowledgment (vs hours of waiting)
✅ Comprehensive Multi-Issue Handling
🔒 Privacy-First: All data stays local with Ollama
💰 Cost-Effective: No per-token charges or usage limits


## Cell 9b [added]: Measured results vs. the lab's claims
Here quality and time are **measured** (the original hard-codes "100% success"), and the business numbers are checked.

In [14]:
results = [result1, result2, result3]
is_empty = lambda text: not str(text or "").strip()

print("📏 QUALITY MEASURED IN THIS RUN (instead of a hard-coded 100%)")
print("=" * 60)
print(f"{'':10}{'original pipeline':>24}{'fixed pipeline (v2)':>24}")
for i, (old, new) in enumerate(zip(results, fixed_results), 1):
    old_status = "EMPTY research" if is_empty(old["research"]) else "complete"
    new_status = "EMPTY research" if is_empty(new["research"]) else "complete"
    print(f"Ticket {i}: {old_status:>22} {old['elapsed_seconds']:>4.0f}s {new_status:>17} {new['elapsed_seconds']:>4.0f}s")
old_ok = sum(not is_empty(r["research"]) and not is_empty(r["response"]) for r in results)
new_ok = sum(not is_empty(r["research"]) and not is_empty(r["response"]) for r in fixed_results)
print(f"Complete tickets: original {old_ok}/3 | fixed {new_ok}/3")
print(f"Group chat: original {'FAILED (' + collab_error.split(':')[0] + ')' if collab_error else 'ok'} "
      f"after {collab_seconds:.0f}s | fixed ok in {collab_v2_seconds:.0f}s")

measured = [r["elapsed_seconds"] for r in fixed_results]
avg_seconds = sum(measured) / len(measured)
print(f"\nAverage time per ticket (fixed pipeline): {avg_seconds:.1f} s (lab assumed 45 s)")

# Same formula as Cell 9, but with the measured time
manual_minutes, daily_tickets, work_days, hourly_rate, implementation = 30, 150, 250, 35, 25_000
saved_minutes = manual_minutes - avg_seconds / 60
annual_hours = saved_minutes * daily_tickets / 60 * work_days
annual_savings = annual_hours * hourly_rate

print("\n💰 BUSINESS CASE WITH MEASURED TIME (same assumptions as Cell 9)")
print("=" * 45)
print(f"Annual hours saved  : {annual_hours:,.0f}")
print(f"Annual savings      : ${annual_savings:,.0f}")
print(f"ROI on $25K         : {annual_savings / implementation:.1f}x")

print("\n⚠️ CONSISTENCY CHECK OF THE LAB'S CLAIMS")
print("=" * 45)
print(f"Headline claim '$65K+ annual savings' vs the lab's own formula: ${impact_metrics['annual_cost_savings']:,.0f}")
print("   -> the two figures differ by ~10x; the $639K comes from assuming 100% of the 30 manual minutes")
print("      per ticket disappears. A realistic case keeps a human review step (see below).")
print("'$85,000 problem' vs '$639K savings': savings cannot exceed the cost of the problem.")
print("'Success Rate: 100%' is hard-coded in Cell 9; it was not measured.")

review_minutes = 5  # a human still reviews/sends every AI draft
realistic_hours = (manual_minutes - review_minutes - avg_seconds / 60) * daily_tickets / 60 * work_days
print(f"\nWith a {review_minutes}-minute human review per ticket: ~${realistic_hours * hourly_rate:,.0f}/year,")
print("still assuming the full 150 tickets/day are handled at 30 minutes each today.")

📏 QUALITY MEASURED IN THIS RUN (instead of a hard-coded 100%)
                 original pipeline     fixed pipeline (v2)
Ticket 1:         EMPTY research   10s          complete   18s
Ticket 2:         EMPTY research    7s          complete   17s
Ticket 3:         EMPTY research    3s          complete   18s
Complete tickets: original 0/3 | fixed 3/3
Group chat: original ok after 426s | fixed ok in 32s

Average time per ticket (fixed pipeline): 18.0 s (lab assumed 45 s)

💰 BUSINESS CASE WITH MEASURED TIME (same assumptions as Cell 9)
Annual hours saved  : 18,563
Annual savings      : $649,700
ROI on $25K         : 26.0x

⚠️ CONSISTENCY CHECK OF THE LAB'S CLAIMS
Headline claim '$65K+ annual savings' vs the lab's own formula: $639,844
   -> the two figures differ by ~10x; the $639K comes from assuming 100% of the 30 manual minutes
      per ticket disappears. A realistic case keeps a human review step (see below).
'$85,000 problem' vs '$639K savings': savings cannot exceed the cost of th

## Cell 10: Portfolio project summary (original code)

In [15]:
def project_summary():
    """Showcase what we've built and its portfolio value"""

    achievements = {
        "AutoGen Skills Demonstrated": [
            "Multi-agent system architecture design",
            "Specialized agent roles and responsibilities",
            "Sequential workflow automation",
            "Group chat management for complex scenarios",
            "Local AI deployment with Ollama integration"
        ],

        "Business Value Created": [
            "73 hours daily time savings across support team",
            "$65,000+ annual cost reduction",
            "98% faster response times (5 min vs 5 hours)",
            "24/7 automated ticket processing capability",
            "Zero ongoing API costs with local Ollama",
            "Complete data privacy with local processing"
        ],

        "Technical Capabilities": [
            "Intelligent ticket triage and categorization",
            "Automated solution research and documentation",
            "Professional customer response generation",
            "Multi-issue ticket handling with agent collaboration",
            "Local AI deployment without external dependencies"
        ]
    }

    print("🎯 SUPPORTFLOW PROJECT SUMMARY")
    print("=" * 40)

    for category, items in achievements.items():
        print(f"\n✅ {category}:")
        for item in items:
            print(f"   • {item}")

    print(f"\n🚀 PORTFOLIO PRESENTATION:")
    portfolio_pitch = """
    'I built SupportFlow using AutoGen with local Ollama llama3.2 - a multi-agent
    customer support system that automates ticket triage, solution research, and
    response generation.

    The system processes tickets 98% faster than manual handling, saving
    $65,000 annually while providing 24/7 automated support capability
    with zero ongoing API costs using local AI.

    Key technical achievements:
    • 3 specialized agents with distinct responsibilities
    • Sequential and collaborative workflow management
    • Local AI deployment with enterprise-grade privacy
    • Measurable ROI with production-ready architecture'
    """
    print(portfolio_pitch)

    return achievements

# Display project summary
summary = project_summary()

🎯 SUPPORTFLOW PROJECT SUMMARY

✅ AutoGen Skills Demonstrated:
   • Multi-agent system architecture design
   • Specialized agent roles and responsibilities
   • Sequential workflow automation
   • Group chat management for complex scenarios
   • Local AI deployment with Ollama integration

✅ Business Value Created:
   • 73 hours daily time savings across support team
   • $65,000+ annual cost reduction
   • 98% faster response times (5 min vs 5 hours)
   • 24/7 automated ticket processing capability
   • Zero ongoing API costs with local Ollama
   • Complete data privacy with local processing

✅ Technical Capabilities:
   • Intelligent ticket triage and categorization
   • Automated solution research and documentation
   • Professional customer response generation
   • Multi-issue ticket handling with agent collaboration
   • Local AI deployment without external dependencies

🚀 PORTFOLIO PRESENTATION:

    'I built SupportFlow using AutoGen with local Ollama llama3.2 - a multi-agent
  

In [16]:
# [added] Save all agent outputs to a JSON file
with open("supportflow_results.json", "w", encoding="utf-8") as f:
    json.dump({"original_pipeline": results,
               "fixed_pipeline": fixed_results,
               "original_group_chat": {"error": collab_error, "seconds": round(collab_seconds, 1)},
               "fixed_group_chat": {"seconds": round(collab_v2_seconds, 1),
                                    "messages": [{"name": m.get("name"), "content": m.get("content")}
                                                 for m in groupchat_v2.messages]}},
              f, indent=2, ensure_ascii=False)
print("Saved supportflow_results.json")

Saved supportflow_results.json
